In [1]:
%load_ext autoreload
%autoreload 3

import warnings
import pandas as pd
import numpy as np
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
from gencost.crosswalk import Crosswalk
from gencost.waterfall import DataBySubplant
import sklearn
import sqlite3
import datetime
from datetime import datetime as dt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
pd.set_option('display.max_columns',None)

warnings.simplefilter(action="once")

/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from gencost.constants import (
    COLS_FOR_REGRESSION,
    CURRENT_EP_COLS,
    FILL_IN_EP_COLS,
    FOSSIL_PRIME_MOVER_MAP,
    FUEL_GROUP_MAP,
    GET_860_GEN_COLS,
    HIST_EP_COLS,
    
)


In [2]:
xwalk = Crosswalk()
# self to be able to copy / paste from waterfall.py for dev ease
self = DataBySubplant(xwalk)

try to figure out what's going on

In [7]:
hist_data = self.get_historical_by_generator()

xwalk = self.xwalk

df_860 = self.get_860_by_x(subplant_id_col="generator_id")

df_923_cf = self.get_gf923_by_generator(counterfactuals=True)

# list of cols we need for melt

"""
missing data type #1: find generators that don't repeat for complete year range
and spit out plant / gen / missing year / prime / fuel

prime and fuel based on latest reported (and highest mmbtu) PF
observation in allocated GF923

"""
_h = hist_data[["plant_id_eia", "generator_id"]].drop_duplicates()

missing_years = (
    pd.concat(
        _h.assign(report_date=rd) for rd in hist_data.report_date.unique()
    )
    .merge(
        hist_data[["plant_id_eia", "generator_id", "report_date"]],
        on=["plant_id_eia", "generator_id", "report_date"],
        how="outer",
        indicator="exists",
    )
    .query('exists == "left_only"')
    .assign(fuss=lambda x: "missing_years")
    .merge(
        df_923_cf.groupby(
            [
                "plant_id_eia",
                "generator_id",
                pd.Grouper(key="report_date", freq="YS"),
                "prime_mover_code",
                "fuel_group",
            ]
        )
        .agg({"mmbtu": "sum", "net_mwh": "sum"})
        .reset_index()  # keep latest prime fuel observation in gf 923 (largest single fuel)
        # .query('report_date == "2020-01-01"')
        # .query("mmbtu > 0 & net_mwh > 0") do we want to put prime/fuel of gens in GF  # noqa: W505
        # reporting zeros?
        .sort_values(
            by=["plant_id_eia", "generator_id", "report_date", "mmbtu"],
            ascending=True,
        ).drop_duplicates(subset=["plant_id_eia", "generator_id"], keep="last")[
            [
                "plant_id_eia",
                "generator_id",
                # "report_date",
                "prime_mover_code",
                "fuel_group",
            ]
        ],
        on=["plant_id_eia", "generator_id"],
        how="left",
    )
    .assign(
        prime_mover=lambda x: x.prime_mover_code.replace(
            FOSSIL_PRIME_MOVER_MAP
        ),
        # age_in_current_year=lambda x: 1,
    )
    .merge(
        df_860.sort_values(
            by=["plant_id_eia", "generator_id", "report_date"], ascending=True
        ).drop_duplicates(subset=["plant_id_eia", "generator_id"])[
            [
                "plant_id_eia",
                "generator_id",
                # "report_date",
                "utility_id_eia",
                "respondent_id",
                "respondent_id_purchaser",
                "state",
                "final_ba_code",
                "generator_operating_date",
                "final_respondent_id",
                "balancing_authority_code_eia",
                # "age_in_current_year", not sure what we wanna do about age in this scenario  # noqa: W505
            ]
        ],
        on=["plant_id_eia", "generator_id"],
        how="left",
        validate="m:1",
        # indicator=True,
    )
    .drop_duplicates(
        subset=[
            "plant_id_eia",
            "generator_id",
            "report_date",
            "prime_mover",
            "fuel_group",
        ]
    )
)[
    [
        "plant_id_eia",
        "generator_id",
        "utility_id_eia",
        "respondent_id",
        "respondent_id_purchaser",
        "state",
        "report_date",
        "fuss",
        "prime_mover",
        "fuel_group",
        "final_ba_code",
        # "age_in_current_year",
        "generator_operating_date",
        "final_respondent_id",
        "balancing_authority_code_eia",
    ]
]

/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:267: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(


In [14]:
single_fuel_switch = (
            hist_data.pipe(self.filter_to_single_fuel_generators)
            .assign(
                n_fuels=lambda x: x.groupby(["plant_id_eia", "generator_id"])[
                    "mmbtu"
                ].transform("nunique")
            )
            .query("n_fuels > 1")
            .assign(
                fuss=lambda x: "fuel_switch",
                # year=lambda x: x["report_date"].dt.year,
                fuel=lambda x: x.groupby(["plant_id_eia", "generator_id"])[
                    "mmbtu"
                ].transform("last"),
            )
            .query("mmbtu != fuel")
            .assign(fuel_group=lambda x: x["fuel"].str.replace("_mmbtu", ""))
            .merge(
                xwalk[["plant_id_eia", "generator_id", "prime_mover", "fuel_group"]],
                on=["plant_id_eia", "generator_id", "fuel_group"],
                how="left",
            )
            .drop_duplicates(
                subset=["plant_id_eia", "generator_id", "prime_mover", "fuel_group"]
    )
)
"""
Missing data type #3: identify plant, gen, prime, fuel observations
reporting zero net gen and fuel consumption

"""
zero_reported = (
    df_923_cf.assign(
        n_fuels=lambda x: x.groupby(
            ["plant_id_eia", "generator_id", "report_date"]
        )["energy_source_code_num"].transform("nunique")
    )
    .query("n_fuels == 1")
    .drop(columns=["n_fuels"])
    .groupby(
        [
            "plant_id_eia",
            "generator_id",
            pd.Grouper(key="report_date", freq="YS"),
            "prime_mover_code",
            "fuel_group",
        ]
    )
    .agg({"net_mwh": "sum", "mmbtu": "sum"})
    .reset_index()
    .query(
        'net_mwh == 0 & mmbtu == 0 & report_date >= "2006-01-01" & report_date <= "2020-01-01"'
    )
    .assign(
        prime_mover=lambda x: x.prime_mover_code.replace(
            FOSSIL_PRIME_MOVER_MAP
        ),
        # year=lambda x: x["report_date"].dt.year,
        fuss="zeroes",
    )
    # .drop(columns=["percent_of_gen", "single_fuel", "single_fuel_present"])
)
zero_and_fuel_switch = pd.concat([single_fuel_switch, zero_reported]).merge(
    df_860[
        [
            "plant_id_eia",
            "generator_id",
            "report_date",
            "utility_id_eia",
            "respondent_id",
            "respondent_id_purchaser",
            "state",
            "final_ba_code",
            "generator_operating_date",
            "final_respondent_id",
            "balancing_authority_code_eia",
            # "age_in_current_year", not sure what we w
        ]
    ],
    on=["plant_id_eia", "generator_id", "report_date"],
    how="left",
    # indicator=True,
)


In [63]:
df_860.query('plant_id_eia == 168 & generator_id == "1"',engine='python')

,utility_id_eia,balancing_authority_code_eia,state,plant_id_eia,generator_id,report_date,capacity_mw,prime_mover,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_in_current_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,generator_operating_date,technology_description,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code
352,814,MISO,AR,168,1,2001-01-01,69.0,ST,1,1,1,1,1,1,1,1,1,1,1,1,49.672827,70.587269,20.914442,11.494942,0.0,1951-05-01,Natural Gas Steam Turbine,8,<NA>,8,ETR
15855,814,MISO,AR,168,1,2002-01-01,69.0,ST,1,1,1,1,1,1,1,1,1,1,1,1,50.672142,70.587269,19.915127,12.494258,0.0,1951-05-01,Natural Gas Steam Turbine,8,<NA>,8,ETR
31940,814,MISO,AR,168,1,2003-01-01,69.0,ST,1,1,1,1,1,1,1,1,1,1,1,1,51.671458,70.587269,18.915811,13.493573,0.0,1951-05-01,Natural Gas Steam Turbine,8,<NA>,8,ETR
48367,814,MISO,AR,168,1,2004-01-01,69.0,ST,1,1,1,1,1,1,1,1,1,1,1,1,52.670773,70.587269,17.916496,14.492889,0.0,1951-05-01,Natural Gas Steam Turbine,8,<NA>,8,ETR
64857,814,MISO,AR,168,1,2005-01-01,69.0,ST,1,1,1,1,1,1,1,1,1,1,1,1,53.672827,70.587269,16.914442,15.494942,0.0,1951-05-01,Natural Gas Steam Turbine,8,<NA>,8,ETR
81412,814,MISO,AR,168,1,2006-01-01,69.0,ST,1,1,1,1,1,1,1,1,1,1,1,1,54.672142,70.587269,15.915127,16.494258,0.0,1951-05-01,Natural Gas Steam Turbine,8,<NA>,8,ETR
98110,814,MISO,AR,168,1,2007-01-01,69.0,ST,1,1,1,1,1,1,1,1,1,1,1,1,55.671458,70.587269,14.915811,17.493573,0.0,1951-05-01,Natural Gas Steam Turbine,8,<NA>,8,ETR
115239,814,MISO,AR,168,1,2008-01-01,69.0,ST,1,1,1,1,1,1,1,1,1,1,1,1,56.670773,70.587269,13.916496,18.492889,0.0,1951-05-01,Natural Gas Steam Turbine,8,<NA>,8,ETR
132708,814,MISO,AR,168,1,2009-01-01,69.0,ST,1,1,1,1,1,1,1,1,1,1,1,1,57.672827,70.587269,12.914442,19.494942,0.0,1951-05-01,Natural Gas Steam Turbine,8,<NA>,8,ETR
150416,814,MISO,AR,168,1,2010-01-01,69.0,ST,1,1,1,1,1,1,1,1,1,1,1,1,58.672142,70.587269,11.915127,20.494258,0.0,1951-05-01,Natural Gas Steam Turbine,8,<NA>,8,ETR


In [64]:
missing_years.query('plant_id_eia == 168 & generator_id == "1"',engine='python')

,plant_id_eia,generator_id,utility_id_eia,respondent_id,respondent_id_purchaser,state,report_date,fuss,prime_mover,fuel_group,final_ba_code,generator_operating_date,final_respondent_id,balancing_authority_code_eia
64,168,1,814,8,<NA>,AR,2019-01-01,missing_years,ST,petroleum,ETR,1951-05-01,8,MISO
6117,168,1,814,8,<NA>,AR,2001-01-01,missing_years,ST,petroleum,ETR,1951-05-01,8,MISO
84404,168,1,814,8,<NA>,AR,2013-01-01,missing_years,ST,petroleum,ETR,1951-05-01,8,MISO
90188,168,1,814,8,<NA>,AR,2014-01-01,missing_years,ST,petroleum,ETR,1951-05-01,8,MISO
95934,168,1,814,8,<NA>,AR,2015-01-01,missing_years,ST,petroleum,ETR,1951-05-01,8,MISO
101847,168,1,814,8,<NA>,AR,2016-01-01,missing_years,ST,petroleum,ETR,1951-05-01,8,MISO
107811,168,1,814,8,<NA>,AR,2017-01-01,missing_years,ST,petroleum,ETR,1951-05-01,8,MISO
113758,168,1,814,8,<NA>,AR,2018-01-01,missing_years,ST,petroleum,ETR,1951-05-01,8,MISO
119713,168,1,814,8,<NA>,AR,2020-01-01,missing_years,ST,petroleum,ETR,1951-05-01,8,MISO


In [70]:
pd.concat(
        _h.assign(report_date=rd) for rd in hist_data.report_date.unique()
    )

,plant_id_eia,generator_id,report_date
0,1,1,2019-01-01
3,1,2,2019-01-01
6,1,3,2019-01-01
9,1,5,2019-01-01
39,3,1,2019-01-01
...,...,...,...
406837,64552,2,2020-01-01
406889,64592,GEN1,2020-01-01
406891,64592,GEN2,2020-01-01
406893,64592,GEN3,2020-01-01


In [71]:
_h = hist_data[["plant_id_eia", "generator_id"]].drop_duplicates()

In [73]:
(pd.concat(
        _h.assign(report_date=rd) for rd in hist_data.report_date.unique()
    )
    .merge(
        hist_data[["plant_id_eia", "generator_id", "report_date"]],
        on=["plant_id_eia", "generator_id", "report_date"],
        how="outer",
        indicator="exists",
    )
.query('plant_id_eia == 168 & generator_id == "1"'))

,plant_id_eia,generator_id,report_date,exists
273,168,1,2019-01-01,left_only
18145,168,1,2001-01-01,left_only
36017,168,1,2002-01-01,both
53889,168,1,2003-01-01,both
71761,168,1,2004-01-01,both
89633,168,1,2005-01-01,both
107505,168,1,2006-01-01,both
125377,168,1,2007-01-01,both
143249,168,1,2008-01-01,both
161121,168,1,2009-01-01,both


In [77]:
epd.query('plant_id_eia == 168 & generator_id == "1"',engine='python')

,plant_id_eia,report_date,prime_mover,report_year,capacity_mw,gross_cf,generator_starts,pollution_control_costs_per_kw,real_pollution_control_costs_per_kw,wage_scale,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code,state,utility_id_eia,balancing_authority_code_eia,age_of_observation_secular_adj,age_of_observation,age_relative_to_prime_avg,biofuel_fraction,coal_fraction,natural_gas_fraction,other_fraction,other_gas_fraction,petroleum_fraction,petroleum_coke_fraction,minor_fuels_fraction,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_in_current_year,gross_generation_mwh,net_generation_mwh,inflator_to_2021,fuel_starts,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,generator_id,generator_operating_date,technology_description,type
7682,168,2002-01-01,ST,2002,69.0,0.046015,13,0.0,0.0,0.896035,8,<NA>,8,ETR,AR,814,MISO,8.634267,19.915127,12.494258,0.0,0.0,1.000000e+00,0.0,0.000000,0.000000,0.0,0.000000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,50.672142,70.587269,27813.160000,18868.000000,1.438638,13,NaN,NaN,309768.500000,NaN,NaN,0.000000,NaN,NaN,NaN,18868.000000,NaN,NaN,0.000000e+00,NaN,NaN,NaN,27813.160000,NaN,NaN,0.000000,NaN,1,1951-05-01,Natural Gas Steam Turbine,historical
7683,168,2003-01-01,ST,2003,69.0,0.014001,13,0.0,0.0,0.915930,8,<NA>,8,ETR,AR,814,MISO,9.550197,18.915811,13.493573,0.0,0.0,1.000000e+00,0.0,0.000000,0.000000,0.0,0.000000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,51.671458,70.587269,8462.660000,-482.500000,1.406582,13,NaN,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN,NaN,-482.500000,NaN,NaN,0.000000e+00,NaN,0.0,0.0,8462.660000,0.0,0.000000,0.000000,0.0,1,1951-05-01,Natural Gas Steam Turbine,historical
7684,168,2004-01-01,ST,2004,69.0,0.014590,13,0.0,0.0,0.949568,8,<NA>,8,ETR,AR,814,MISO,10.499764,17.916496,14.492889,0.0,0.0,1.000000e+00,0.0,0.000000,0.000000,0.0,0.000000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,52.670773,70.587269,8843.161000,-101.999000,1.370095,13,NaN,NaN,4004.000000,NaN,NaN,NaN,NaN,NaN,NaN,-101.999000,NaN,NaN,NaN,NaN,NaN,NaN,8843.161000,NaN,NaN,NaN,NaN,1,1951-05-01,Natural Gas Steam Turbine,historical
7685,168,2005-01-01,ST,2005,69.0,0.014285,13,0.0,0.0,0.949952,8,<NA>,8,ETR,AR,814,MISO,11.449716,16.914442,15.494942,0.0,0.0,1.000000e+00,0.0,0.000000,0.000000,0.0,0.000000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,53.672827,70.587269,8634.160000,-311.000000,1.325197,13,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,-311.000000,NaN,NaN,NaN,NaN,NaN,NaN,8634.160000,NaN,NaN,NaN,NaN,1,1951-05-01,Natural Gas Steam Turbine,historical
7686,168,2006-01-01,ST,2006,69.0,0.000000,0,0.0,0.0,0.884169,8,<NA>,8,ETR,AR,814,MISO,12.333885,15.915127,16.494258,0.0,0.0,0.000000e+00,0.0,0.000000,0.000000,0.0,0.000000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,54.672142,70.587269,0.000000,-156.499500,1.283785,0,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,-156.499500,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,1,1951-05-01,Natural Gas Steam Turbine,historical
7687,168,2007-01-01,ST,2007,69.0,0.000000,0,0.0,0.0,0.893976,8,<NA>,8,ETR,AR,814,MISO,13.227861,14.915811,17.493573,0.0,0.0,0.000000e+00,0.0,0.000000,0.000000,0.0,0.000000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,55.671458,70.587269,0.000000,-72.500000,1.248232,0,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,-72.500000,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,1,1951-05-01,Natural Gas Steam Turbine,historical
7688,168,2008-01-01,ST,2008,69.0,0.000000,0,0.0,0.0,0.927583,8,<NA>,8,ETR,AR,814,

In [17]:
cf = ( pd.concat([missing_years, zero_and_fuel_switch])
    # drop duplicates and keep first since we don't want zeroes
    .drop_duplicates(
        subset=[
            "plant_id_eia",
            "generator_id",
            "report_date",
            # "prime_mover",
            # "fuel_group",
        ],
        keep="first",
    ).assign(
        age=lambda x: (
            ((pd.datetime.now() - x.generator_operating_date).dt.days) / 365.25
        ).round(2)
    )
)

/var/folders/lw/gjwq5pd52hb01x363x3vyw6c0000gp/T/ipykernel_12061/3199683243.py:14: FutureWarning: The pandas.datetime class is deprecated and will be removed from pandas in a future version. Import from datetime module instead.
  ((pd.datetime.now() - x.generator_operating_date).dt.days) / 365.25


In [28]:
cf.query('plant_id_eia == 126 & generator_id == "RIC6"',engine='python')

,plant_id_eia,generator_id,utility_id_eia,respondent_id,respondent_id_purchaser,state,report_date,fuss,prime_mover,fuel_group,final_ba_code,generator_operating_date,final_respondent_id,balancing_authority_code_eia,mmbtu,fuel_consumption,percent_of_gen,single_fuel,single_fuel_present,n_fuels,fuel,prime_mover_code,net_mwh,age
1273,126,RIC6,<NA>,<NA>,<NA>,<NA>,2019-01-01,zeroes,IC,natural_gas,NaN,NaT,<NA>,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,IC,0.0,NaN


In [27]:
df_860.query('plant_id_eia == 126 & generator_id == "RIC6"',engine='python')

,utility_id_eia,balancing_authority_code_eia,state,plant_id_eia,generator_id,report_date,capacity_mw,prime_mover,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_in_current_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,generator_operating_date,technology_description,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code


fill in ep data process

In [29]:

def create_fill_in_ep_thresholds(self, df):
    bins = [0, 10, 20, 30, 40, 50, 60, 70, 100]

    labels = [1, 2, 3, 4, 5, 6, 7, 8]

    return df.assign(
        essentials=lambda x: x["report_date"].astype(str)
        + "_"
        + x["prime_mover"]
        + "_"
        + x["fuel_group"],
        ba_plus_essentials=lambda x: x["essentials"] + "_" + x["final_ba_code"],
        age_range=lambda x: pd.cut(x["age"], bins=bins, labels=labels),
        ba_plus_age=lambda x: x["ba_plus_essentials"]
        + "_"
        + x["age_range"].astype(str),
)

In [31]:
age_year=2021
reference_date = dt.strptime(f"12-1-{age_year}", "%m-%d-%Y")

In [33]:
xwalk = self.xwalk
# df_860 = self.get_860_by_x(subplant_id_col="generator_id")

historical = (
    self.get_historical_by_generator()
    .merge(
        xwalk[["plant_id_eia", "generator_id", "prime_mover", "fuel_group"]],
        on=["plant_id_eia", "generator_id", "prime_mover"],
        how="left",
    )
    .assign(
        age=lambda x: (
            ((pd.datetime.now() - x.generator_operating_date).dt.days) / 365.25
        ).round(2),
        report_year=lambda x: x.report_date.dt.year,
    )
    # .assign(year=lambda x: x["report_date"].dt.year)
    .pipe(self.create_fill_in_ep_thresholds)
)

missing = (
    self.find_missing_data()
    .pipe(self.create_fill_in_ep_thresholds)
    .assign(
        essentials_present=lambda x: np.where(
            x["essentials"].isin(historical["essentials"]),
            "essentials",
            pd.NA,
        ),
        ba_plus_essentials_present=lambda x: np.where(
            x["ba_plus_essentials"].isin(historical["ba_plus_essentials"]),
            "ba_plus_essentials",
            pd.NA,
        ),
        ba_plus_age_present=lambda x: np.where(
            x["ba_plus_age"].isin(historical["ba_plus_age"]),
            "ba_plus_age",
            pd.NA,
        ),
    )  # make an exception of when there is no ba code or age
    .assign(
        ba_plus_age_present=lambda x: np.where(
            (x["final_ba_code"].isnull()) | (x["age"].isnull()),
            pd.NA,
            x["ba_plus_age_present"],
        ),
        ba_plus_essentials_present=lambda x: np.where(
            (x["final_ba_code"].isnull()),
            pd.NA,
            x["ba_plus_essentials_present"],
        ),
        # fill_in_score=lambda x: np.where(
        # x["ba_plus_age_present"].notnull(), x["ba_plus_age_present"]
        # ),
        match=lambda x: x["ba_plus_age_present"]
        .fillna(x["ba_plus_essentials_present"])
        .fillna(x["essentials_present"]),
    )
)

"""
Fill in part #1: generation and fuel consumption
Merge missing data df with historical data
1) Loop based on unique values in match column
2) Query based on value (list with different kind of matches)
3) Merge
4) Append to a list
5) drop duplicates since they're might be multiple matches

"""

cols = ["ba_plus_age", "ba_plus_essentials", "essentials"]
filled_in = []

for col in cols:
    # core columns we want to get from historical, append column we're going to merge on  # noqa: W505

    # keep plant specific id columns - plant/gen/utility/ba ids (don't want to fill that in with historical)  # noqa: W505
    df = (
        missing[FILL_IN_EP_COLS]
        .query("match == @col")
        .merge(
            historical[HIST_EP_COLS + [col]].drop_duplicates(subset=col),
            on=[col],
            how="inner",
        )
        .drop_duplicates(subset=["plant_id_eia", "generator_id", "report_date"])
    )

    filled_in.append(df)

# remove columns not in historical df for future concat, excpt mtch and fuss
filled_in_hist_cols = (
    pd.concat(filled_in)
    .drop(
        columns=[
            "ba_plus_age",
            "ba_plus_essentials",
            "essentials",
            "fuel_group",
        ]
    )
    .sort_values(
        by=["plant_id_eia", "generator_id", "report_date"], ascending=True
    )
    .query("prime_mover in @FOSSIL_PRIME_MOVER_MAP")
    .assign(report_year=lambda x: x.report_date.dt.year)
)

"""
Now that we have missing data w/
historical generation + fuel consumption
Move on to:
Fill in part #2: capacity + tech cols
Merge filled in data with latest (2020?) 860 info
Fill in part #3: (w)age cols to recalculate
"""

current = (
    filled_in_hist_cols.merge(
        historical[CURRENT_EP_COLS]
        .drop_duplicates(subset=["plant_id_eia", "generator_id"], keep="last")
        .drop(columns=["report_date"]),
        on=["plant_id_eia", "generator_id"],
        how="inner",  # inner merge to only keep generators in get_exa subset
        validate="m:1",
    )
    .assign(  # re-do age calculations
        age_in_report_year=lambda x: (
            x["report_date"] - x["generator_operating_date"]
        ).dt.days
        / 365.25,
        age_in_current_year=lambda x: (
            reference_date - x["generator_operating_date"]
        ).dt.days
        / 365.25,
        age_of_observation=lambda x: (reference_date - x["report_date"]).dt.days
        / 365.25,
        age_relative_to_prime_avg=lambda x: x["age_in_report_year"]
        - x.groupby(["prime_mover"])["age_in_report_year"].transform("mean"),
    )
    # recalc pollution control costs
    .assign(
        pollution_control_costs_per_kw=lambda x: x[
            "real_pollution_control_costs_per_kw"
        ]
        / x["inflator_to_2021"]
    )
    .sort_values(
        by=["plant_id_eia", "generator_id", "report_date"], ascending=True
    )
    .query("prime_mover in @FOSSIL_PRIME_MOVER_MAP")
    .assign(report_year=lambda x: x.report_date.dt.year)
)

/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:267: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/var/folders/lw/gjwq5pd52hb01x363x3vyw6c0000gp/T/ipykernel_12061/350420757.py:13: FutureWarning: The pandas.datetime class is deprecated and will be removed from pandas in a future version. Import from datetime module instead.
  ((pd.datetime.now() - x.generator_operating_date).dt.days) / 365.25
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:267: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the val

In [61]:
missing_years.query('plant_id_eia == 168 & generator_id == "1"',engine='python')

,plant_id_eia,generator_id,utility_id_eia,respondent_id,respondent_id_purchaser,state,report_date,fuss,prime_mover,fuel_group,final_ba_code,generator_operating_date,final_respondent_id,balancing_authority_code_eia
64,168,1,814,8,<NA>,AR,2019-01-01,missing_years,ST,petroleum,ETR,1951-05-01,8,MISO
6117,168,1,814,8,<NA>,AR,2001-01-01,missing_years,ST,petroleum,ETR,1951-05-01,8,MISO
84404,168,1,814,8,<NA>,AR,2013-01-01,missing_years,ST,petroleum,ETR,1951-05-01,8,MISO
90188,168,1,814,8,<NA>,AR,2014-01-01,missing_years,ST,petroleum,ETR,1951-05-01,8,MISO
95934,168,1,814,8,<NA>,AR,2015-01-01,missing_years,ST,petroleum,ETR,1951-05-01,8,MISO
101847,168,1,814,8,<NA>,AR,2016-01-01,missing_years,ST,petroleum,ETR,1951-05-01,8,MISO
107811,168,1,814,8,<NA>,AR,2017-01-01,missing_years,ST,petroleum,ETR,1951-05-01,8,MISO
113758,168,1,814,8,<NA>,AR,2018-01-01,missing_years,ST,petroleum,ETR,1951-05-01,8,MISO
119713,168,1,814,8,<NA>,AR,2020-01-01,missing_years,ST,petroleum,ETR,1951-05-01,8,MISO


In [90]:
df_860.query('plant_id_eia == 345 & generator_id == "3"',engine='python')

,utility_id_eia,balancing_authority_code_eia,state,plant_id_eia,generator_id,report_date,capacity_mw,prime_mover,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_in_current_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,generator_operating_date,technology_description,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code
692,15830,CISO,CA,345,3,2001-01-01,138.0,GT,1,1,1,1,1,1,1,1,1,1,1,1,30.754278,51.66872,20.914442,8.740428,0.0,1970-04-01,Natural Gas Fired Combustion Turbine,<NA>,<NA>,<NA>,CAISO
16187,15830,CISO,CA,345,3,2002-01-01,138.0,GT,1,1,1,1,1,1,1,1,1,1,1,1,31.753593,51.66872,19.915127,9.739743,0.0,1970-04-01,Natural Gas Fired Combustion Turbine,<NA>,<NA>,<NA>,CAISO
32263,15830,CISO,CA,345,3,2003-01-01,138.0,GT,1,1,1,1,1,1,1,1,1,1,1,1,32.752909,51.66872,18.915811,10.739059,0.0,1970-04-01,Natural Gas Fired Combustion Turbine,<NA>,<NA>,<NA>,CAISO
48678,15830,CISO,CA,345,3,2004-01-01,138.0,GT,1,1,1,1,1,1,1,1,1,1,1,1,33.752225,51.66872,17.916496,11.738374,0.0,1970-04-01,Natural Gas Fired Combustion Turbine,<NA>,<NA>,<NA>,CAISO
65168,15830,CISO,CA,345,3,2005-01-01,138.0,GT,1,1,1,1,1,1,1,1,1,1,1,1,34.754278,51.66872,16.914442,12.740428,0.0,1970-04-01,Natural Gas Fired Combustion Turbine,<NA>,<NA>,<NA>,CAISO
81719,15830,CISO,CA,345,3,2006-01-01,138.0,GT,1,1,1,1,1,1,1,1,1,1,1,1,35.753593,51.66872,15.915127,13.739743,0.0,1970-04-01,Natural Gas Fired Combustion Turbine,<NA>,<NA>,<NA>,CAISO
98422,15830,CISO,CA,345,3,2007-01-01,138.0,GT,1,1,1,1,1,1,1,1,1,1,1,1,36.752909,51.66872,14.915811,14.739059,0.0,1970-04-01,Natural Gas Fired Combustion Turbine,<NA>,<NA>,<NA>,CAISO
115551,15830,CISO,CA,345,3,2008-01-01,138.0,GT,1,1,1,1,1,1,1,1,1,1,1,1,37.752225,51.66872,13.916496,15.738374,0.0,1970-04-01,Natural Gas Fired Combustion Turbine,<NA>,<NA>,<NA>,CAISO
133016,15830,CISO,CA,345,3,2009-01-01,138.0,GT,1,1,1,1,1,1,1,1,1,1,1,1,38.754278,51.66872,12.914442,16.740428,0.0,1970-04-01,Natural Gas Fired Combustion Turbine,<NA>,<NA>,<NA>,CAISO
150733,15908,CISO,CA,345,3,2010-01-01,138.0,GT,1,1,1,1,1,1,1,1,1,1,1,1,39.753593,51.66872,11.915127,17.739743,0.0,1970-04-01,Natural Gas Fired Combustion Turbine,<NA>,<NA>,<NA>,CAISO


In [91]:
historical.query('plant_id_eia == 345 & generator_id == "3"')

,plant_id_eia,report_date,prime_mover,report_year,capacity_mw,gross_cf,generator_starts,pollution_control_costs_per_kw,real_pollution_control_costs_per_kw,wage_scale,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code,state,utility_id_eia,balancing_authority_code_eia,age_of_observation_secular_adj,age_of_observation,age_relative_to_prime_avg,biofuel_fraction,coal_fraction,natural_gas_fraction,other_fraction,other_gas_fraction,petroleum_fraction,petroleum_coke_fraction,minor_fuels_fraction,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_in_current_year,gross_generation_mwh,net_generation_mwh,inflator_to_2021,fuel_starts,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,generator_id,generator_operating_date,technology_description,type,mmbtu
14056,345,2001-01-01,GT,2001,138.0,1.734738,50,0.0,0.0,1.223899,<NA>,<NA>,<NA>,CAISO,CA,15830,CISO,9.142052,20.914442,8.740428,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,30.754278,51.66872,2.097091e+06,2.066920e+06,1.461383,50,NaN,NaN,2.040678e+07,NaN,NaN,NaN,NaN,NaN,NaN,2.066920e+06,NaN,NaN,NaN,NaN,NaN,NaN,2.097091e+06,NaN,NaN,NaN,NaN,3,1970-04-01,Natural Gas Fired Combustion Turbine,historical,1.0
14057,345,2002-01-01,GT,2002,138.0,0.228963,50,0.0,0.0,1.118694,<NA>,<NA>,<NA>,CAISO,CA,15830,CISO,10.260746,19.915127,9.739743,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,31.753593,51.66872,2.767886e+05,2.370689e+05,1.438638,50,NaN,NaN,2.334712e+06,NaN,NaN,NaN,NaN,NaN,NaN,2.370689e+05,NaN,NaN,NaN,NaN,NaN,NaN,2.767886e+05,NaN,NaN,NaN,NaN,3,1970-04-01,Natural Gas Fired Combustion Turbine,historical,1.0
14058,345,2003-01-01,GT,2003,138.0,0.017787,50,0.0,0.0,1.159887,<NA>,<NA>,<NA>,CAISO,CA,15830,CISO,11.420633,18.915811,10.739059,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,32.752909,51.66872,2.150195e+04,9.120000e+03,1.406582,50,NaN,NaN,1.510030e+05,NaN,NaN,NaN,NaN,NaN,NaN,9.120000e+03,NaN,NaN,NaN,NaN,NaN,NaN,2.150195e+04,NaN,NaN,NaN,NaN,3,1970-04-01,Natural Gas Fired Combustion Turbine,historical,1.0
14060,345,2005-01-01,GT,2005,138.0,0.011392,50,0.0,0.0,1.207596,<NA>,<NA>,<NA>,CAISO,CA,15830,CISO,13.827080,16.914442,12.740428,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,34.754278,51.66872,1.377195e+04,1.390000e+03,1.325197,50,NaN,NaN,2.641660e+04,NaN,NaN,NaN,NaN,NaN,NaN,1.390000e+03,NaN,NaN,NaN,NaN,NaN,NaN,1.377195e+04,NaN,NaN,NaN,NaN,3,1970-04-01,Natural Gas Fired Combustion Turbine,historical,1.0
14061,345,2006-01-01,GT,2006,138.0,0.012995,50,0.0,0.0,1.218299,<NA>,<NA>,<NA>,CAISO,CA,15830,CISO,15.045379,15.915127,13.739743,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,35.753593,51.66872,1.570995e+04,3.328000e+03,1.283785,50,NaN,NaN,5.324400e+04,NaN,NaN,NaN,NaN,NaN,NaN,3.328000e+03,NaN,NaN,NaN,NaN,NaN,NaN,1.570995e+04,NaN,NaN,NaN,NaN,3,1970-04-01,Natural Gas Fired Combustion Turbine,historical,1.0
14062,345,2007-01-01,GT,2007,138.0,0.046001,50,0.0,0.0,1.205149,<NA>,<NA>,<NA>,CAISO,CA,15830,CISO,16.250528,14.915811,14.739059,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,36.752909,51.66872,5.560995e+04,4.322800e+04,1.248232,50,NaN,NaN,4.545040e+05,NaN,NaN,NaN,NaN,NaN,NaN,4.322800e+04,NaN,NaN,NaN,NaN,NaN,NaN,5.560995e+04,NaN,NaN,NaN,NaN,3,1970-04-01,Natural Gas Fired Combustion Turbine,historical,1.0
14063,345,2008-01-01,GT,2008,138.0,0.011294,50,0.0,0.

In [39]:

historical = self.get_historical_by_generator().assign(
    type=lambda x: "historical",
    mmbtu=lambda x: x.filter(like="_fraction").sum(axis=1),
)
# take out where mmbtu and net gen are zero, for cf to replace
historical_clean = historical.loc[
    ~((historical["mmbtu"] == 0) & (historical["net_generation_mwh"] == 0))
]

cf = self.fill_in_ep_data().assign(type=lambda x: "counterfactual")

/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:267: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:2905: FutureWarning: The pandas.datetime class is deprecated and will be removed from pandas in a future version. Import from datetime module instead.
  ((pd.datetime.now() - x.generator_operating_date).dt.days) / 365.25
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:267: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` wi

In [44]:
epd = pd.read_parquet('/Users/mcastillo/Documents/GitHub/gencost/temp_output/epd.parquet')

In [92]:
epd.query('plant_id_eia == 345 & generator_id == "3"',engine='python')

,plant_id_eia,report_date,prime_mover,report_year,capacity_mw,gross_cf,generator_starts,pollution_control_costs_per_kw,real_pollution_control_costs_per_kw,wage_scale,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code,state,utility_id_eia,balancing_authority_code_eia,age_of_observation_secular_adj,age_of_observation,age_relative_to_prime_avg,biofuel_fraction,coal_fraction,natural_gas_fraction,other_fraction,other_gas_fraction,petroleum_fraction,petroleum_coke_fraction,minor_fuels_fraction,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_in_current_year,gross_generation_mwh,net_generation_mwh,inflator_to_2021,fuel_starts,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,generator_id,generator_operating_date,technology_description,type
14056,345,2001-01-01,GT,2001,138.0,1.734738,50,0.0,0.0,1.223899,<NA>,<NA>,<NA>,CAISO,CA,15830,CISO,9.142052,20.914442,8.740428,0.0,0.0,1.000000,0.0,0.0,0.000000,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,30.754278,51.66872,2.097091e+06,2.066920e+06,1.461383,50,NaN,NaN,2.040678e+07,NaN,NaN,NaN,NaN,NaN,NaN,2.066920e+06,NaN,NaN,NaN,NaN,NaN,NaN,2.097091e+06,NaN,NaN,NaN,NaN,3,1970-04-01,Natural Gas Fired Combustion Turbine,historical
14057,345,2002-01-01,GT,2002,138.0,0.228963,50,0.0,0.0,1.118694,<NA>,<NA>,<NA>,CAISO,CA,15830,CISO,10.260746,19.915127,9.739743,0.0,0.0,1.000000,0.0,0.0,0.000000,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,31.753593,51.66872,2.767886e+05,2.370689e+05,1.438638,50,NaN,NaN,2.334712e+06,NaN,NaN,NaN,NaN,NaN,NaN,2.370689e+05,NaN,NaN,NaN,NaN,NaN,NaN,2.767886e+05,NaN,NaN,NaN,NaN,3,1970-04-01,Natural Gas Fired Combustion Turbine,historical
14058,345,2003-01-01,GT,2003,138.0,0.017787,50,0.0,0.0,1.159887,<NA>,<NA>,<NA>,CAISO,CA,15830,CISO,11.420633,18.915811,10.739059,0.0,0.0,1.000000,0.0,0.0,0.000000,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,32.752909,51.66872,2.150195e+04,9.120000e+03,1.406582,50,NaN,NaN,1.510030e+05,NaN,NaN,NaN,NaN,NaN,NaN,9.120000e+03,NaN,NaN,NaN,NaN,NaN,NaN,2.150195e+04,NaN,NaN,NaN,NaN,3,1970-04-01,Natural Gas Fired Combustion Turbine,historical
14060,345,2005-01-01,GT,2005,138.0,0.011392,50,0.0,0.0,1.207596,<NA>,<NA>,<NA>,CAISO,CA,15830,CISO,13.827080,16.914442,12.740428,0.0,0.0,1.000000,0.0,0.0,0.000000,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,34.754278,51.66872,1.377195e+04,1.390000e+03,1.325197,50,NaN,NaN,2.641660e+04,NaN,NaN,NaN,NaN,NaN,NaN,1.390000e+03,NaN,NaN,NaN,NaN,NaN,NaN,1.377195e+04,NaN,NaN,NaN,NaN,3,1970-04-01,Natural Gas Fired Combustion Turbine,historical
14061,345,2006-01-01,GT,2006,138.0,0.012995,50,0.0,0.0,1.218299,<NA>,<NA>,<NA>,CAISO,CA,15830,CISO,15.045379,15.915127,13.739743,0.0,0.0,1.000000,0.0,0.0,0.000000,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,35.753593,51.66872,1.570995e+04,3.328000e+03,1.283785,50,NaN,NaN,5.324400e+04,NaN,NaN,NaN,NaN,NaN,NaN,3.328000e+03,NaN,NaN,NaN,NaN,NaN,NaN,1.570995e+04,NaN,NaN,NaN,NaN,3,1970-04-01,Natural Gas Fired Combustion Turbine,historical
14062,345,2007-01-01,GT,2007,138.0,0.046001,50,0.0,0.0,1.205149,<NA>,<NA>,<NA>,CAISO,CA,15830,CISO,16.250528,14.915811,14.739059,0.0,0.0,1.000000,0.0,0.0,0.000000,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,36.752909,51.66872,5.560995e+04,4.322800e+04,1.248232,50,NaN,NaN,4.545040e+05,NaN,NaN,NaN,NaN,NaN,NaN,4.322800e+04,NaN,NaN,NaN,NaN,NaN,NaN,5.560995e+04,NaN,NaN,NaN,NaN,3,1970-04-01,Natural Gas Fired Combustion Turbine,historical
14063,345,2008-01-01,GT

In [57]:
epd = (
            pd.concat([historical_clean, cf])
            .drop(columns=["mmbtu"])
            .drop_duplicates(
                subset=["plant_id_eia", "generator_id", "report_date"], keep="last"
            )
        )


In [96]:
historical.query('plant_id_eia == 377 & generator_id == "3"')

,plant_id_eia,report_date,prime_mover,report_year,capacity_mw,gross_cf,generator_starts,pollution_control_costs_per_kw,real_pollution_control_costs_per_kw,wage_scale,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code,state,utility_id_eia,balancing_authority_code_eia,age_of_observation_secular_adj,age_of_observation,age_relative_to_prime_avg,biofuel_fraction,coal_fraction,natural_gas_fraction,other_fraction,other_gas_fraction,petroleum_fraction,petroleum_coke_fraction,minor_fuels_fraction,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_in_current_year,gross_generation_mwh,net_generation_mwh,inflator_to_2021,fuel_starts,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,generator_id,generator_operating_date,technology_description,type,mmbtu
14693,377,2001-01-01,ST,2001,20.0,0.057200,13,108.003487,157.834503,1.223899,601,108,601,LDWP,CA,7294,LDWP,9.142052,20.914442,8.907673,0.315975,0.0,0.684025,0.0,0.0,0.0,0.0,0.315975,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,47.085558,68.0,10021.389756,7428.589756,1.461383,13,26247.050754,NaN,56819.728493,NaN,NaN,NaN,NaN,2586.934156,NaN,4841.655600,NaN,NaN,NaN,NaN,3166.511666,NaN,6854.878090,NaN,NaN,NaN,NaN,3,1953-12-01,Natural Gas Steam Turbine,historical,1.315975
14694,377,2002-01-01,ST,2002,20.0,0.036675,13,108.003487,155.377935,1.118694,601,108,601,LDWP,CA,7294,LDWP,10.260746,19.915127,9.906989,0.639479,0.0,0.360521,0.0,0.0,0.0,0.0,0.639479,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,48.084873,68.0,6425.406778,3832.606778,1.438638,13,32417.318244,NaN,18276.035627,NaN,NaN,NaN,NaN,2506.035665,NaN,1326.571112,NaN,NaN,NaN,NaN,4108.910547,NaN,2316.496231,NaN,NaN,NaN,NaN,3,1953-12-01,Natural Gas Steam Turbine,historical,1.639479
14695,377,2003-01-01,ST,2003,20.0,0.150942,13,108.003487,151.915709,1.159887,601,108,601,LDWP,CA,7294,LDWP,11.420633,18.915811,10.906304,0.506125,0.0,0.493875,0.0,0.0,0.0,0.0,0.506125,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,49.084189,68.0,26445.022222,23852.222222,1.406582,13,182523.703704,NaN,178105.925926,NaN,NaN,0.0,NaN,12598.333333,NaN,11253.888889,NaN,NaN,0.0,NaN,13384.489249,NaN,13060.532974,NaN,NaN,0.0,NaN,3,1953-12-01,Natural Gas Steam Turbine,historical,1.506125
14696,377,2004-01-01,ST,2004,20.0,0.160636,13,108.003487,147.975069,1.198851,601,108,601,LDWP,CA,7294,LDWP,12.619484,17.916496,11.905620,0.454547,0.0,0.545453,0.0,0.0,0.0,0.0,0.454547,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,50.083504,68.0,28220.577778,25627.777778,1.370095,13,170496.481481,NaN,204594.259259,NaN,NaN,0.0,NaN,12251.296296,NaN,13376.481481,NaN,NaN,0.0,NaN,12827.587285,NaN,15392.990493,NaN,NaN,0.0,NaN,3,1953-12-01,Natural Gas Steam Turbine,historical,1.454547
14697,377,2005-01-01,ST,2005,20.0,0.167315,13,108.003487,143.125911,1.207596,601,108,601,LDWP,CA,7294,LDWP,13.827080,16.914442,12.907673,0.513262,0.0,0.486738,0.0,0.0,0.0,0.0,0.513262,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,51.085558,68.0,29313.540741,26720.740741,1.325197,13,200554.705556,NaN,190190.603704,NaN,NaN,0.0,NaN,13496.111111,NaN,13224.629630,NaN,NaN,0.0,NaN,15045.525545,NaN,14268.015196,NaN,NaN,0.0,NaN,3,1953-12-01,Natural Gas Steam Turbine,historical,1.513262
14698,377,2006-01-01,ST,2006,20.0,0.170525,13,108.003487,138.653227,1.218299,601,108,601,LDWP,CA,7294,LDWP,15.045379,15.915127,13.906989,0.529892,0.0,0.470108,0.0,0.0,0.0,0.0,0.529892,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,52.084873,68.0,29875.948148,27283.148148,1.283785,13,207787.962963,N

In [145]:
cf.query('plant_id_eia == 491 & generator_id == "3"')

,plant_id_eia,report_date,prime_mover,report_year,capacity_mw,gross_cf,generator_starts,pollution_control_costs_per_kw,real_pollution_control_costs_per_kw,wage_scale,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code,state,utility_id_eia,balancing_authority_code_eia,age_of_observation_secular_adj,age_of_observation,age_relative_to_prime_avg,biofuel_fraction,coal_fraction,natural_gas_fraction,other_fraction,other_gas_fraction,petroleum_fraction,petroleum_coke_fraction,minor_fuels_fraction,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_in_current_year,gross_generation_mwh,net_generation_mwh,inflator_to_2021,fuel_starts,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,generator_id,generator_operating_date,technology_description,type


In [146]:
epd.query('plant_id_eia == 491 & generator_id == "3"',engine='python')

,plant_id_eia,report_date,prime_mover,report_year,capacity_mw,gross_cf,generator_starts,pollution_control_costs_per_kw,real_pollution_control_costs_per_kw,wage_scale,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code,state,utility_id_eia,balancing_authority_code_eia,age_of_observation_secular_adj,age_of_observation,age_relative_to_prime_avg,biofuel_fraction,coal_fraction,natural_gas_fraction,other_fraction,other_gas_fraction,petroleum_fraction,petroleum_coke_fraction,minor_fuels_fraction,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_in_current_year,gross_generation_mwh,net_generation_mwh,inflator_to_2021,fuel_starts,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,generator_id,generator_operating_date,technology_description,type


In [ ]:
epd.query('plant_id_eia ')

In [ ]:
epd = (
            pd.concat([historical_clean, cf])
            .drop(columns=["mmbtu"])
            .drop_duplicates(
                subset=["plant_id_eia", "generator_id", "report_date"], keep="last"
            )
        )

In [46]:
from gencost import epd_with_vom_fom_som

In [93]:
test = epd_with_vom_fom_som.get_epd_w_vom_fom_som()

/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:267: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-

DataBySubplant QC
DataBySubplant QC results: pass
New data QC
New data QC results: pass


/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:208: FutureWarning: Support for multi-dimensional indexing (e.g. `obj[:, None]`) is deprecated and will be removed in a future version.  Convert to a numpy array before indexing instead.
  np.repeat(df[old_cols].sum(axis=1)[:, np.newaxis], len(old_cols), 1) != 0.0,
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:213: FutureWarning: Support for multi-dimensional indexing (e.g. `obj[:, None]`) is deprecated and will be removed in a future version.  Convert to a numpy array before indexing instead.
  df[old_cols].isna().sum(axis=1)[:, np.newaxis], len(old_cols), 1
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:222: FutureWarning: Support for multi-dimensional indexing (e.g. `obj[:, None]`) is deprecated and will be removed in a future version.  Convert to a numpy array before indexing

DataBySubplant QC
DataBySubplant QC results: pass
New data QC
New data QC results: pass


/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:2538: FutureWarning: Passing a dict as an indexer is deprecated and will raise in a future version. Use a list instead.
  df = schema.validate(df[core_columns | gen_columns])
/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/pandera/backends/pandas/base.py:120: UserWarning: <Schema DataFrameSchema(
    columns={
        'plant_id_eia': <Schema Column(name=plant_id_eia, type=DataType(int64))>
        'report_date': <Schema Column(name=report_date, type=DataType(datetime64[ns]))>
        'prime_mover': <Schema Column(name=prime_mover, type=DataType(str))>
        'report_year': <Schema Column(name=report_year, type=DataType(int64))>
        'capacity_mw': <Schema Column(name=capacity_mw, type=DataType(float64))>
        'gross_cf': <Schema Column(name=gross_cf, type=DataType(float64))>
        'generator_starts': <Schema Column(name=generator_starts, type=DataType(int64))>
        

In [139]:
df_860.query('plant_id_eia == 468 & generator_id == "1"',engine='python')

,utility_id_eia,balancing_authority_code_eia,state,plant_id_eia,generator_id,report_date,capacity_mw,prime_mover,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_in_current_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,generator_operating_date,technology_description,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code


In [172]:
historical.query('plant_id_eia == 8063 & generator_id == "1"',engine='python')

,plant_id_eia,report_date,prime_mover,report_year,capacity_mw,gross_cf,generator_starts,pollution_control_costs_per_kw,real_pollution_control_costs_per_kw,wage_scale,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code,state,utility_id_eia,balancing_authority_code_eia,age_of_observation_secular_adj,age_of_observation,age_relative_to_prime_avg,biofuel_fraction,coal_fraction,natural_gas_fraction,other_fraction,other_gas_fraction,petroleum_fraction,petroleum_coke_fraction,minor_fuels_fraction,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_in_current_year,gross_generation_mwh,net_generation_mwh,inflator_to_2021,fuel_starts,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,generator_id,generator_operating_date,technology_description,type,mmbtu
206023,8063,2001-01-01,ST,2001,799.2,0.364525,13,0.0,0.0,1.000282,<NA>,<NA>,<NA>,ERCO,TX,19323,ERCO,7.689292,20.914442,-12.505058,0.0,0.0,0.969777,0.0,0.0,0.030223,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,25.672827,46.587269,2.552037e+06,2.377310e+06,1.461383,13,NaN,NaN,2.301903e+07,NaN,NaN,717396.684647,NaN,NaN,NaN,2.317455e+06,NaN,NaN,59855.022822,NaN,NaN,NaN,2.474906e+06,NaN,NaN,77131.347672,NaN,1,1975-05-01,Natural Gas Steam Turbine,historical,1.0
206024,8063,2002-01-01,ST,2002,799.2,0.325211,13,0.0,0.0,0.995906,<NA>,<NA>,<NA>,ERCO,TX,19323,ERCO,8.685198,19.915127,-11.505742,0.0,0.0,0.996599,0.0,0.0,0.003401,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,26.672142,46.587269,2.276797e+06,2.102070e+06,1.438638,13,NaN,NaN,1.716310e+07,NaN,NaN,58570.692946,NaN,NaN,NaN,2.095465e+06,NaN,NaN,6605.421162,NaN,NaN,NaN,2.269054e+06,NaN,NaN,7743.361447,NaN,1,1975-05-01,Natural Gas Steam Turbine,historical,1.0
206025,8063,2003-01-01,ST,2003,799.2,0.257980,13,0.0,0.0,0.991148,<NA>,<NA>,<NA>,ERCO,TX,19323,ERCO,9.676346,18.915811,-10.506427,0.0,0.0,0.999537,0.0,0.0,0.000463,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,27.671458,46.587269,1.806113e+06,1.631386e+06,1.406582,13,NaN,NaN,1.545925e+07,NaN,NaN,7155.000000,NaN,NaN,NaN,1.630645e+06,NaN,NaN,741.000000,NaN,NaN,NaN,1.805278e+06,NaN,NaN,835.536158,NaN,1,1975-05-01,Natural Gas Steam Turbine,historical,1.0
206026,8063,2004-01-01,ST,2004,799.2,0.041067,13,0.0,0.0,1.000185,<NA>,<NA>,<NA>,ERCO,TX,19323,ERCO,10.676531,17.916496,-9.507111,0.0,0.0,0.999484,0.0,0.0,0.000516,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,28.670773,46.587269,2.882999e+05,1.971950e+05,1.370095,13,NaN,NaN,2.604957e+06,NaN,NaN,1346.000000,NaN,NaN,NaN,1.978440e+05,NaN,NaN,-649.000000,NaN,NaN,NaN,2.881510e+05,NaN,NaN,148.889714,NaN,1,1975-05-01,Natural Gas Steam Turbine,historical,1.0
206027,8063,2005-01-01,ST,2005,799.2,0.067692,13,0.0,0.0,1.047514,<NA>,<NA>,<NA>,ERCO,TX,19323,ERCO,11.724045,16.914442,-8.505058,0.0,0.0,1.000000,0.0,0.0,0.000000,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,29.672827,46.587269,4.739103e+05,2.438810e+05,1.325197,13,NaN,NaN,3.236949e+06,NaN,NaN,0.000000,NaN,NaN,NaN,2.438810e+05,NaN,NaN,0.000000,NaN,NaN,NaN,4.739103e+05,NaN,NaN,0.000000,NaN,1,1975-05-01,Natural Gas Steam Turbine,historical,1.0
206028,8063,2006-01-01,ST,2006,799.2,0.008022,16,0.0,0.0,1.067586,<NA>,<NA>,<NA>,ERCO,TX,19323,ERCO,12.791631,15.915127,-7.505742,0.0,0.0,1.000000,0.0,0.0,0.000000,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,30.672142,46.587269,5.616000e+04,5.168900e+04,1.283785,16,NaN,NaN,7.053000e+05,NaN,NaN,0.000000,NaN,NaN,NaN,5.168900e+04,NaN,NaN,0.000000,NaN,NaN,NaN,5.61

In [171]:
test.query('plant_id_eia == 8063 & generator_id == "1"',engine='python')

,plant_id_eia,report_date,prime_mover,report_year,capacity_mw,gross_cf,generator_starts,pollution_control_costs_per_kw,real_pollution_control_costs_per_kw,wage_scale,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code,state,utility_id_eia,balancing_authority_code_eia,age_of_observation_secular_adj,age_of_observation,age_relative_to_prime_avg,biofuel_fraction,coal_fraction,natural_gas_fraction,other_fraction,other_gas_fraction,petroleum_fraction,petroleum_coke_fraction,minor_fuels_fraction,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_in_current_year,gross_generation_mwh,net_generation_mwh,inflator_to_2021,fuel_starts,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,generator_id,generator_operating_date,technology_description,type,rowid,cls,fom,vom,som,om_per_mwh,_merge
117261,8063,2001-01-01,ST,2001,799.2,0.400486,13,0.0,0.0,1.000282,<NA>,<NA>,<NA>,ERCO,TX,19323,ERCO,7.689292,20.914442,-12.505058,0.0,0.0,0.969777,0.0,0.0,0.030223,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,25.672827,46.587269,2.803796e+06,2.377310e+06,1.461383,13,NaN,NaN,2.301903e+07,NaN,NaN,717396.684647,NaN,NaN,NaN,2.317455e+06,NaN,NaN,59855.022822,NaN,NaN,NaN,2.719056e+06,NaN,NaN,84740.378007,NaN,1,1975-05-01,Natural Gas Steam Turbine,historical,117218.0,3.0,20.798116,-14.526821,-0.259148,-11.524616,both
117262,8063,2002-01-01,ST,2002,799.2,0.361171,13,0.0,0.0,0.995906,<NA>,<NA>,<NA>,ERCO,TX,19323,ERCO,8.685198,19.915127,-11.505742,0.0,0.0,0.996599,0.0,0.0,0.003401,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,26.672142,46.587269,2.528556e+06,2.102070e+06,1.438638,13,NaN,NaN,1.716310e+07,NaN,NaN,58570.692946,NaN,NaN,NaN,2.095465e+06,NaN,NaN,6605.421162,NaN,NaN,NaN,2.519957e+06,NaN,NaN,8599.591622,NaN,1,1975-05-01,Natural Gas Steam Turbine,historical,117219.0,3.0,20.798116,-13.496217,-0.266260,-10.321613,both
117263,8063,2003-01-01,ST,2003,799.2,0.293940,13,0.0,0.0,0.991148,<NA>,<NA>,<NA>,ERCO,TX,19323,ERCO,9.676346,18.915811,-10.506427,0.0,0.0,0.999537,0.0,0.0,0.000463,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,27.671458,46.587269,2.057872e+06,1.631386e+06,1.406582,13,NaN,NaN,1.545925e+07,NaN,NaN,7155.000000,NaN,NaN,NaN,1.630645e+06,NaN,NaN,741.000000,NaN,NaN,NaN,2.056920e+06,NaN,NaN,952.003899,NaN,1,1975-05-01,Natural Gas Steam Turbine,historical,117220.0,3.0,20.798116,-12.053786,-0.265887,-8.494439,both
117264,8063,2004-01-01,ST,2004,799.2,0.046923,13,0.0,0.0,1.000185,<NA>,<NA>,<NA>,ERCO,TX,19323,ERCO,10.676531,17.916496,-9.507111,0.0,0.0,0.999484,0.0,0.0,0.000516,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,28.670773,46.587269,3.294065e+05,1.971950e+05,1.370095,13,NaN,NaN,2.604957e+06,NaN,NaN,1346.000000,NaN,NaN,NaN,1.978440e+05,NaN,NaN,-649.000000,NaN,NaN,NaN,3.292364e+05,NaN,NaN,170.118854,NaN,1,1975-05-01,Natural Gas Steam Turbine,historical,117221.0,2.0,23.483866,9.345686,-0.110473,85.481859,both
117265,8063,2005-01-01,ST,2005,799.2,0.053720,13,0.0,0.0,1.047514,<NA>,<NA>,<NA>,ERCO,TX,19323,ERCO,11.724045,16.914442,-8.505058,0.0,0.0,1.000000,0.0,0.0,0.000000,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,29.672827,46.587269,3.760925e+05,2.438810e+05,1.325197,13,NaN,NaN,3.236949e+06,NaN,NaN,0.000000,NaN,NaN,NaN,2.438810e+05,NaN,NaN,0.000000,NaN,NaN,NaN,3.760925e+05,NaN,NaN,0.000000,NaN,1,1975-05-01,Natural Gas Steam Turbine,historical,117222.0,2.0,24.387418,10.622527,-0.115713,80.566680,both
117266,8063,2006-01-01,ST,2006,799.2,0.008022,16,0.0,0.0,1.067586,

In [167]:
historical.query('plant_id_eia == 2341',engine='python')

,plant_id_eia,report_date,prime_mover,report_year,capacity_mw,gross_cf,generator_starts,pollution_control_costs_per_kw,real_pollution_control_costs_per_kw,wage_scale,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code,state,utility_id_eia,balancing_authority_code_eia,age_of_observation_secular_adj,age_of_observation,age_relative_to_prime_avg,biofuel_fraction,coal_fraction,natural_gas_fraction,other_fraction,other_gas_fraction,petroleum_fraction,petroleum_coke_fraction,minor_fuels_fraction,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_in_current_year,gross_generation_mwh,net_generation_mwh,inflator_to_2021,fuel_starts,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,generator_id,generator_operating_date,technology_description,type,mmbtu
96047,2341,2001-01-01,ST,2001,818.1,0.740222,10,0.0,0.0,1.128554,161,<NA>,161,CAISO,NV,17609,CISO,8.682437,20.914442,-8.422922,0.0,0.997071,0.002929,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,29.754962,50.669405,5.304844e+06,5.125985e+06,1.461383,10,NaN,5.152693e+07,151363.000,NaN,NaN,NaN,NaN,NaN,5.109930e+06,16054.50000,NaN,NaN,NaN,NaN,NaN,5.289307e+06,15537.610542,NaN,NaN,NaN,NaN,1,1971-04-01,Conventional Steam Coal,historical,1.0
96048,2341,2002-01-01,ST,2002,818.1,0.734519,10,0.0,0.0,1.070744,161,<NA>,161,CAISO,NV,17609,CISO,9.753181,19.915127,-7.423607,0.0,0.997849,0.002151,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,30.754278,50.669405,5.263974e+06,5.085115e+06,1.438638,10,NaN,4.230559e+07,91208.500,NaN,NaN,NaN,NaN,NaN,5.075690e+06,9425.00000,NaN,NaN,NaN,NaN,NaN,5.252650e+06,11324.420985,NaN,NaN,NaN,NaN,1,1971-04-01,Conventional Steam Coal,historical,1.0
96049,2341,2003-01-01,ST,2003,818.1,0.701538,10,0.0,0.0,1.090294,161,<NA>,161,CAISO,NV,17609,CISO,10.843475,18.915811,-6.424291,0.0,0.997542,0.002458,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,31.753593,50.669405,5.027609e+06,4.848750e+06,1.406582,10,NaN,4.887658e+07,120413.500,NaN,NaN,NaN,NaN,NaN,4.836920e+06,11830.50000,NaN,NaN,NaN,NaN,NaN,5.015253e+06,12355.698361,NaN,NaN,NaN,NaN,1,1971-04-01,Conventional Steam Coal,historical,1.0
96050,2341,2004-01-01,ST,2004,818.1,0.734193,10,0.0,0.0,1.092785,161,<NA>,161,CAISO,NV,17609,CISO,11.936260,17.916496,-5.424976,0.0,0.998254,0.001746,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,32.752909,50.669405,5.276050e+06,5.097191e+06,1.370095,10,NaN,5.082042e+07,88900.500,NaN,NaN,NaN,NaN,NaN,5.087121e+06,10070.00000,NaN,NaN,NaN,NaN,NaN,5.266837e+06,9213.312333,NaN,NaN,NaN,NaN,1,1971-04-01,Conventional Steam Coal,historical,1.0
96051,2341,2005-01-01,ST,2005,818.1,0.760068,10,0.0,0.0,1.127245,161,<NA>,161,CAISO,NV,17609,CISO,13.063505,16.914442,-4.422922,0.0,0.998213,0.001787,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,33.754962,50.669405,5.447067e+06,5.268208e+06,1.325197,10,NaN,5.360118e+07,95945.595,NaN,NaN,NaN,NaN,NaN,5.258848e+06,9359.00000,NaN,NaN,NaN,NaN,NaN,5.437334e+06,9732.775123,NaN,NaN,NaN,NaN,1,1971-04-01,Conventional Steam Coal,historical,1.0
96052,2341,2006-01-01,ST,2006,818.1,0.000000,0,0.0,0.0,1.123075,161,<NA>,161,CAISO,NV,17609,CISO,14.186580,15.915127,-3.423607,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,34.754278,50.669405,0.000000e+00,-1.890800e+04,1.283785,0,NaN,0.000000e+00,0.000,NaN,NaN,NaN,NaN,NaN,-1.890800e+04,0.00000,NaN,NaN,NaN,NaN,0.0,0.000000e+00,0.000000,0.0,0.0,0.0,0.

In [113]:
df_860.query('plant_id_eia == 491',engine='python')

,utility_id_eia,balancing_authority_code_eia,state,plant_id_eia,generator_id,report_date,capacity_mw,prime_mover,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_in_current_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,generator_operating_date,technology_description,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code
941,3227,PSCO,CO,491,3,2001-01-01,0.5,IC,1,1,1,1,1,1,1,1,1,1,1,1,37.505818,58.420260,20.914442,15.639296,0.0,1963-07-01,Petroleum Liquids,<NA>,<NA>,<NA>,PSCO
942,3227,PSCO,CO,491,5,2001-01-01,1.0,IC,1,1,1,1,1,1,1,1,1,1,1,1,41.420945,62.335387,20.914442,19.554423,0.0,1959-08-01,Petroleum Liquids,<NA>,<NA>,<NA>,PSCO
16438,3227,PSCO,CO,491,3,2002-01-01,0.5,IC,1,1,1,1,1,1,1,1,1,1,1,1,38.505133,58.420260,19.915127,16.638612,0.0,1963-07-01,Petroleum Liquids,<NA>,<NA>,<NA>,PSCO
16439,3227,PSCO,CO,491,5,2002-01-01,1.0,IC,1,1,1,1,1,1,1,1,1,1,1,1,42.420260,62.335387,19.915127,20.553739,0.0,1959-08-01,Petroleum Liquids,<NA>,<NA>,<NA>,PSCO
32511,3227,PSCO,CO,491,3,2003-01-01,0.5,IC,1,1,1,1,1,1,1,1,1,1,1,1,39.504449,58.420260,18.915811,17.637928,0.0,1963-07-01,Petroleum Liquids,<NA>,<NA>,<NA>,PSCO
32512,3227,PSCO,CO,491,5,2003-01-01,1.0,IC,1,1,1,1,1,1,1,1,1,1,1,1,43.419576,62.335387,18.915811,21.553054,0.0,1959-08-01,Petroleum Liquids,<NA>,<NA>,<NA>,PSCO
48928,3227,PSCO,CO,491,3,2004-01-01,0.5,IC,1,1,1,1,1,1,1,1,1,1,1,1,40.503765,58.420260,17.916496,18.637243,0.0,1963-07-01,Petroleum Liquids,<NA>,<NA>,<NA>,PSCO
48929,3227,PSCO,CO,491,5,2004-01-01,1.0,IC,1,1,1,1,1,1,1,1,1,1,1,1,44.418891,62.335387,17.916496,22.552370,0.0,1959-08-01,Petroleum Liquids,<NA>,<NA>,<NA>,PSCO
65424,3227,PSCO,CO,491,3,2005-01-01,0.5,IC,1,1,1,1,1,1,1,1,1,1,1,1,41.505818,58.420260,16.914442,19.639296,0.0,1963-07-01,Petroleum Liquids,<NA>,<NA>,<NA>,PSCO
65425,3227,PSCO,CO,491,5,2005-01-01,1.0,IC,1,1,1,1,1,1,1,1,1,1,1,1,45.420945,62.335387,16.914442,23.554423,0.0,1959-08-01,Petroleum Liquids,<NA>,<NA>,<NA>,PSCO


In [159]:
df_860.query('plant_id_eia == 1403 & capacity_mw == 895.1',engine='python')

,utility_id_eia,balancing_authority_code_eia,state,plant_id_eia,generator_id,report_date,capacity_mw,prime_mover,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_in_current_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,generator_operating_date,technology_description,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code
3027,11241,MISO,LA,1403,5,2001-01-01,895.1,ST,1,1,1,1,1,1,1,1,1,1,1,1,27.586585,48.501027,20.914442,-10.591300,151.799728,1973-06-01,Natural Gas Steam Turbine,454,<NA>,454,ETR
18578,11241,MISO,LA,1403,5,2002-01-01,895.1,ST,1,1,1,1,1,1,1,1,1,1,1,1,28.585900,48.501027,19.915127,-9.591984,151.799728,1973-06-01,Natural Gas Steam Turbine,454,<NA>,454,ETR
34623,11241,MISO,LA,1403,5,2003-01-01,895.1,ST,1,1,1,1,1,1,1,1,1,1,1,1,29.585216,48.501027,18.915811,-8.592669,151.799728,1973-06-01,Natural Gas Steam Turbine,454,<NA>,454,ETR
50987,11241,MISO,LA,1403,5,2004-01-01,895.1,ST,1,1,1,1,1,1,1,1,1,1,1,1,30.584531,48.501027,17.916496,-7.593353,151.799728,1973-06-01,Natural Gas Steam Turbine,454,<NA>,454,ETR
67477,11241,MISO,LA,1403,5,2005-01-01,895.1,ST,1,1,1,1,1,1,1,1,1,1,1,1,31.586585,48.501027,16.914442,-6.591300,151.799728,1973-06-01,Natural Gas Steam Turbine,454,<NA>,454,ETR
84016,11241,MISO,LA,1403,5,2006-01-01,895.1,ST,1,1,1,1,1,1,1,1,1,1,1,1,32.585900,48.501027,15.915127,-5.591984,151.799728,1973-06-01,Natural Gas Steam Turbine,454,<NA>,454,ETR
100710,11241,MISO,LA,1403,5,2007-01-01,895.1,ST,1,1,1,1,1,1,1,1,1,1,1,1,33.585216,48.501027,14.915811,-4.592669,151.799728,1973-06-01,Natural Gas Steam Turbine,454,<NA>,454,ETR
117817,11241,MISO,LA,1403,5,2008-01-01,895.1,ST,1,1,1,1,1,1,1,1,1,1,1,1,34.584531,48.501027,13.916496,-3.593353,151.799728,1973-06-01,Natural Gas Steam Turbine,454,<NA>,454,ETR
135283,11241,MISO,LA,1403,5,2009-01-01,895.1,ST,1,1,1,1,1,1,1,1,1,1,1,1,35.586585,48.501027,12.914442,-2.591300,151.799728,1973-06-01,Natural Gas Steam Turbine,454,<NA>,454,ETR
152981,11241,MISO,LA,1403,5,2010-01-01,895.1,ST,1,1,1,1,1,1,1,1,1,1,1,1,36.585900,48.501027,11.915127,-1.591984,151.799728,1973-06-01,Natural Gas Steam Turbine,454,<NA>,454,ETR


In [153]:
historical.query('plant_id_eia == 1403')['generator_id'].unique()

array(['1', '2', '3', '5', '6A', '6B', '6C'], dtype=object)

In [152]:
historical.query('plant_id_eia == 1403 & generator_id == "6"',engine='python')

,plant_id_eia,report_date,prime_mover,report_year,capacity_mw,gross_cf,generator_starts,pollution_control_costs_per_kw,real_pollution_control_costs_per_kw,wage_scale,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code,state,utility_id_eia,balancing_authority_code_eia,age_of_observation_secular_adj,age_of_observation,age_relative_to_prime_avg,biofuel_fraction,coal_fraction,natural_gas_fraction,other_fraction,other_gas_fraction,petroleum_fraction,petroleum_coke_fraction,minor_fuels_fraction,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_in_current_year,gross_generation_mwh,net_generation_mwh,inflator_to_2021,fuel_starts,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,generator_id,generator_operating_date,technology_description,type,mmbtu


In [104]:
cf.query('plant_id_eia == 468 & generator_id == "1"')

,plant_id_eia,report_date,prime_mover,report_year,capacity_mw,gross_cf,generator_starts,pollution_control_costs_per_kw,real_pollution_control_costs_per_kw,wage_scale,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code,state,utility_id_eia,balancing_authority_code_eia,age_of_observation_secular_adj,age_of_observation,age_relative_to_prime_avg,biofuel_fraction,coal_fraction,natural_gas_fraction,other_fraction,other_gas_fraction,petroleum_fraction,petroleum_coke_fraction,minor_fuels_fraction,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_in_current_year,gross_generation_mwh,net_generation_mwh,inflator_to_2021,fuel_starts,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,generator_id,generator_operating_date,technology_description,type


In [102]:
test.query('plant_id_eia == 468 & generator_id == "1"')

,plant_id_eia,report_date,prime_mover,report_year,capacity_mw,gross_cf,generator_starts,pollution_control_costs_per_kw,real_pollution_control_costs_per_kw,wage_scale,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code,state,utility_id_eia,balancing_authority_code_eia,age_of_observation_secular_adj,age_of_observation,age_relative_to_prime_avg,biofuel_fraction,coal_fraction,natural_gas_fraction,other_fraction,other_gas_fraction,petroleum_fraction,petroleum_coke_fraction,minor_fuels_fraction,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_in_current_year,gross_generation_mwh,net_generation_mwh,inflator_to_2021,fuel_starts,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,generator_id,generator_operating_date,technology_description,type,rowid,cls,fom,vom,som,om_per_mwh,_merge


get historical by gen

In [115]:
from gencost import predict_parasitic_load
from gencost.package_data import PACKAGE_PATH
import logging
logger = logging.getLogger(__name__)

def allocate_col_by(
    df: pd.DataFrame,
    *,
    to_allocate: str,
    new_suffix: str,
    old_suffix: str,
    fillna: int | float | str | None = None,
    rollup_by: list | None = None,
    drop: bool = True,
    drop_bad_rows: str | None = None,
):
    """Allocate a column proportionally using values in a set of columns.

    Args:
        df: input dataframe
        to_allocate: the column that will be allocated
        new_suffix: suffix that will replace the old one in the new columns
        old_suffix: suffix of columns to use for allocation
        fillna: fill nans in new columns with new value
        rollup_by: columns to use in groupby, this rollup is used for allocations
            when a given row has only nans and more than one zero.
        drop: drop the old_suffix columns
        drop_bad_rows: drop rows where the allocation failed, rows are dropped if
            argument is not None, pass a string to insert into log message

    Returns:

    """
    old_cols = list(df.filter(like=old_suffix).columns)
    new_cols = [x.replace(old_suffix, new_suffix) for x in old_cols]
    if rollup_by is not None:
        agg_old_cols = df.groupby(rollup_by)[old_cols].transform("sum")
        multi_zeros = agg_old_cols.divide(agg_old_cols.sum(axis=1), axis=0)
    else:
        multi_zeros = 0.0
    df[new_cols] = np.multiply(
        np.where(
            # this checks where row sums to zero, have to do this at the row level to
            # make sure allocation is consistent across row
            np.repeat(df[old_cols].sum(axis=1)[:, np.newaxis], len(old_cols), 1) != 0.0,
            df[old_cols].divide(df[old_cols].sum(axis=1), axis=0),
            np.where(
                # all columns nan except one that is zero, zero col gets 100% allocation
                np.repeat(
                    df[old_cols].isna().sum(axis=1)[:, np.newaxis], len(old_cols), 1
                )
                == len(old_cols) - 1,
                # if only one zero, zero col gets 100% allocation
                np.where(df[old_cols] == 0.0, 1.0, np.nan),
                # otherwise use allocation based on all years of data
                multi_zeros,
            ),
        ),
        df[to_allocate][:, np.newaxis],
    )
    if fillna is not None:
        df[new_cols] = df[new_cols].fillna(fillna)
    if drop_bad_rows is not None:
        close = np.isclose(df[to_allocate], df[new_cols].sum(axis=1), rtol=1e-2)
        if (num := np.sum(~close)) > 0:
            logger.warning(
                "%s: dropping %s rows because %s allocation by %s failed.",
                drop_bad_rows,
                num,
                to_allocate,
                old_suffix,
            )
            df = df[close]
    if drop:
        return df.drop(columns=old_cols)
    return df

def drop_zero_cols(df, keep=("solid_fuel_gasification",)):
    """Drop numeric columns that sum to zero."""
    non_zeros = df.sum(axis=0, numeric_only=True) != 0
    for col in keep:
        non_zeros.loc[col] = True
    return df.loc[:, [non_zeros.get(x, True) for x in df.columns]]


get historical by gen

In [116]:
df_860 = self.get_860_by_x(subplant_id_col="generator_id")
df_923 = self.get_gf923_by_generator()
df_cems = self.get_cems_by_generator()
data_by_subplant = self.get_exa_by_subplant()

# merge cems and 923 for gen-fuel level allocation of gross gen
df = pd.merge(
    df_923,
    df_cems,
    on=["plant_id_eia", "generator_id", "report_date"],
    how="outer",
    validate="1:1",
    indicator="cems_923_merge",  # merge in cols for regression
).merge(
    df_860[COLS_FOR_REGRESSION],
    on=["plant_id_eia", "generator_id", "report_date"],
    validate="1:1",
    how="left",
)

# fill in gross gen for gens not in CEMS & non zero net gen
df_w_predictions = (
    predict_parasitic_load.predict_parasitic_load(data_by_subplant, df)
    .assign(
        predicted_gross_generation_mwh=lambda x: x["parasitic_load"]
        * (x["capacity_mw"] * 8760)
        + x["net_generation_mwh"],
        gross_generation_mwh=lambda x: np.where(
            (x["cems_923_merge"] == "left_only")
            & (abs(x["net_generation_mwh"]) > 0)
            & (x["net_generation_mwh"].notna()),
            x["predicted_gross_generation_mwh"],
            x["gross_generation_mwh"],
        ),  # predict using mean starts by tech
        predicted_generator_starts=lambda x: x.groupby(
            "technology_description"
        )["generator_starts"].transform("median"),
        predicted_fuel_starts=lambda x: x.groupby("technology_description")[
            "generator_starts"
        ].transform("median"),
        generator_starts=lambda x: np.where(
            (x["cems_923_merge"] == "left_only")
            & (abs(x["net_generation_mwh"]) > 0)
            & (x["net_generation_mwh"].notna()),
            x["predicted_generator_starts"],
            x["generator_starts"],
        ),
        fuel_starts=lambda x: np.where(
            (x["cems_923_merge"] == "left_only")
            & (abs(x["net_generation_mwh"]) > 0)
            & (x["net_generation_mwh"].notna()),
            x["predicted_generator_starts"],
            x["generator_starts"],
        ),
        # overwrite merge indicator to keep everything all the way through
        cems_923_merge=lambda x: "both",
    )
    .drop(  # drop what we needed for and got from regression
        columns=[
            "predicted_gross_generation_mwh",
            "predicted_generator_starts",
            "predicted_fuel_starts",
            "prime_mover",
            "state",
            "capacity_mw",
            "associated_combined_heat_power",
            "duct_burners",
            "bypass_heat_recovery",
            "solid_fuel_gasification",
            "carbon_capture",
            "fluidized_bed_tech",
            "pulverized_coal_tech",
            "stoker_tech",
            "other_combustion_tech",
            "subcritical_tech",
            "supercritical_tech",
            "ultrasupercritical_tech",
            "age_in_report_year",
            "age_in_current_year",
            "age_of_observation",
            "age_relative_to_prime_avg",
            "pollution_control_costs_per_kw",
            "technology_description",
        ]
    )
)

# allocate cems gross gen using pivoted gf 923
cems_and_923 = allocate_col_by(
    df_w_predictions,
    to_allocate="gross_generation_mwh",
    new_suffix="_gross_mwh",
    old_suffix="_mmbtu",
    rollup_by=["plant_id_eia", "generator_id"],
    drop=False,
)

cems_and_923 = drop_zero_cols(cems_and_923)

merged = (
    cems_and_923.assign(
        generator_starts=lambda x: np.where(
            (x["generator_starts"].isna()), 0, x["generator_starts"]
        ),
        fuel_starts=lambda x: np.where(
            (x["fuel_starts"].isna()), 0, x["fuel_starts"]
        ),
    )
    .merge(
        df_860,
        on=["plant_id_eia", "generator_id", "report_date"],
        validate="1:1",
        how="outer",
        indicator="exa_merge",
    )
    .assign(
        _merge=lambda x: x[["cems_923_merge", "exa_merge"]]
        .astype("string")
        .fillna("")
        .agg(",".join, axis=1)
        .replace(
            {
                "both,both": "all",
                "both,left_only": "cems_923_only",
                "right_only,both": "exa_860",
            }
        )
    )
    .drop(columns=["cems_923_merge", "exa_merge"])
    .merge(
        self.get_wage_scale(),
        on=["report_date", "state"],
        how="left",
        validate="m:1",
    )
    .fillna({"wage_scale": 1})
    .assign(
        hrs_in_yr=lambda x: np.where(
            x.report_date.dt.is_leap_year, 8784, 8760
        ),
        gross_cf=lambda x: x.gross_generation_mwh
        / (x.capacity_mw * x.hrs_in_yr),
    )
    .merge(
        pd.read_parquet(
            PACKAGE_PATH
            / "860_FERC_matching_cost_regressions.parquet.gzip",
        )[["report_year", "inflator_to_2021"]]
        .drop_duplicates()
        .assign(
            report_date=lambda x: pd.to_datetime(
                x["report_year"], format="%Y"
            )
        ),
        on=["report_date"],
        how="left",
    )
    .assign(
        real_pollution_control_costs_per_kw=lambda x: x.pollution_control_costs_per_kw
        * x.inflator_to_2021
    )  # pandera caught 3 observations w/null prime movers
    .query("prime_mover.notnull()")
    .query("prime_mover in @FOSSIL_PRIME_MOVER_MAP")
    # fix pandera issue for nan starts
)

new = (
    predict_parasitic_load.predict_parasitic_load(data_by_subplant, merged)
    .assign(
        predicted_gross_generation_mwh=lambda x: x["parasitic_load"]
        * (x["capacity_mw"] * 8760)
        + x["net_generation_mwh"],
        gross_generation_mwh=lambda x: np.where(
            (x["gross_cf"] > 1.5)
            | (
                (x["gross_generation_mwh"] == 0)
                & abs(x["net_generation_mwh"])
                > 0 & (x["net_generation_mwh"].notna())
            )
            | x["gross_generation_mwh"]
            < 0,
            x["predicted_gross_generation_mwh"],
            x["gross_generation_mwh"],
        ),
        gross_gen_value=lambda x: np.where(
            (x["gross_cf"] > 1.5)
            | (
                (x["gross_generation_mwh"] == 0)
                & abs(x["net_generation_mwh"])
                > 0 & (x["net_generation_mwh"].notna())
            )
            | x["gross_generation_mwh"]
            < 0,
            "predicted",
            "reported",
        ),
    )  # two plants from 2001 with 0 mw cap in 860
    .query("capacity_mw > 0")
    .assign(
        gross_cf=lambda x: x.gross_generation_mwh
        / (x.capacity_mw * x.hrs_in_yr),
    )
).drop(columns="predicted_gross_generation_mwh")

# fuel fraction calcs from merge all
gross_mwh_cols = new.filter(like="_gross_mwh").columns

new[[c.replace("_gross_mwh", "_fraction") for c in gross_mwh_cols]] = (
    new[gross_mwh_cols]
    .divide(new[gross_mwh_cols].sum(axis=1), axis=0)
    .fillna(0.0)
)

core_fuels = ["coal_fraction", "natural_gas_fraction", "petroleum_fraction"]

out = (
    new.assign(
        minor_fuels_fraction=lambda x: x.filter(like="_fraction").sum(
            axis=1
        )
        - x[core_fuels].sum(axis=1)
    )
    .query('_merge == "all"')
    .drop(columns=["_merge", "hrs_in_yr"])
    .query(
        "gross_generation_mwh.notna()"
    )  # only 2 generators, mismatch in subset
    .assign(report_year=lambda x: x.report_date.dt.year)
    # drop 2021 values, not in previous version
    .query("report_year <= 2020")
    .query("gross_generation_mwh >= 0")
)


/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:267: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk


DataBySubplant QC
DataBySubplant QC results: pass
New data QC
New data QC results: pass


/var/folders/lw/gjwq5pd52hb01x363x3vyw6c0000gp/T/ipykernel_12061/389322797.py:45: FutureWarning: Support for multi-dimensional indexing (e.g. `obj[:, None]`) is deprecated and will be removed in a future version.  Convert to a numpy array before indexing instead.
  np.repeat(df[old_cols].sum(axis=1)[:, np.newaxis], len(old_cols), 1) != 0.0,
/var/folders/lw/gjwq5pd52hb01x363x3vyw6c0000gp/T/ipykernel_12061/389322797.py:50: FutureWarning: Support for multi-dimensional indexing (e.g. `obj[:, None]`) is deprecated and will be removed in a future version.  Convert to a numpy array before indexing instead.
  df[old_cols].isna().sum(axis=1)[:, np.newaxis], len(old_cols), 1
/var/folders/lw/gjwq5pd52hb01x363x3vyw6c0000gp/T/ipykernel_12061/389322797.py:59: FutureWarning: Support for multi-dimensional indexing (e.g. `obj[:, None]`) is deprecated and will be removed in a future version.  Convert to a numpy array before indexing instead.
  df[to_allocate][:, np.newaxis],
/opt/anaconda3/envs/gencost/

DataBySubplant QC
DataBySubplant QC results: pass
New data QC
New data QC results: pass


In [150]:
new.query('plant_id_eia == 491',engine='python')

,plant_id_eia,generator_id,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,renew_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,nuclear_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,renew_net_mwh,net_generation_mwh,generator_starts,fuel_starts,gross_generation_mwh,parasitic_load,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,nuclear_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,renew_gross_mwh,utility_id_eia,balancing_authority_code_eia,state,capacity_mw,prime_mover,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_in_current_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,generator_operating_date,technology_description,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code,_merge,wage_scale,age_of_observation_secular_adj,hrs_in_yr,gross_cf,report_year,inflator_to_2021,real_pollution_control_costs_per_kw,gross_gen_value,biofuel_fraction,coal_fraction,natural_gas_fraction,nuclear_fraction,other_fraction,other_gas_fraction,petroleum_fraction,petroleum_coke_fraction,renew_fraction
19523,491,3,2001-01-01,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0,0.0,0.0,NaN,0.013431,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3227,PSCO,CO,0.5,IC,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,37.505818,58.420260,20.914442,15.639296,0.0,1963-07-01,Petroleum Liquids,<NA>,<NA>,<NA>,PSCO,all,0.990621,7.890396,8760,NaN,2001.0,1.461383,0.0,reported,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19524,491,3,2002-01-01,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0,0.0,0.0,NaN,0.013431,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3227,PSCO,CO,0.5,IC,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,38.505133,58.420260,19.915127,16.638612,0.0,1963-07-01,Petroleum Liquids,<NA>,<NA>,<NA>,PSCO,all,0.965335,8.855731,8760,NaN,2002.0,1.438638,0.0,reported,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19525,491,3,2003-01-01,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0,0.0,0.0,NaN,0.013431,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3227,PSCO,CO,0.5,IC,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,39.504449,58.420260,18.915811,17.637928,0.0,1963-07-01,Petroleum Liquids,<NA>,<NA>,<NA>,PSCO,all,0.963219,9.81895,8760,NaN,2003.0,1.406582,0.0,reported,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19526,491,3,2004-01-01,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0,0.0,0.0,NaN,0.013431,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3227,PSCO,CO,0.5,IC,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,40.503765,58.420260,17.916496,18.637243,0.0,1963-07-01,Petroleum Liquids,<NA>,<NA>,<NA>,PSCO,all,1.020642,10.839593,8784,NaN,2004.0,1.370095,0.0,reported,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19527,491,3,2005-01-01,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0,0.0,0.0,NaN,0.013431,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3227,PSCO,CO,0.5,IC,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,41.505818,58.420260,16.914442,19.639296,0.0,1963-07-01,Petroleum Liquids,<NA>,<NA>,<NA>,PSCO,all,0.94066,11.780252,8760,NaN,2005.0,1.325197,0.0,reported,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19528,491,3,2006-01-01,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0,0.0,0.0,NaN,0.013431,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3227,PSCO,CO,0.5,IC,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,42.505133,58.420260,15.915127,20.638612,0.0,1963-07-01,Petroleum Liquids,<NA>,<NA>,<NA>,PSCO,all,0.937463,12.717715,8760,NaN,2006.0,1.283785,0.0,reported,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19529,491,3,2007-01-01,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.

get 860 by gen

In [117]:
from etoolbox.utils.pudl_helpers import (
    month_year_to_date,
    simplify_columns,
    sum_and_weighted_average_agg,
)

In [122]:
coi = (
pd.read_parquet(PACKAGE_PATH / "unit_level_costs_with_flag.parquet.gzip")
.pipe(simplify_columns)
.pipe(month_year_to_date)
.rename(columns={"plant_id": "plant_id_eia"})
)[["plant_id_eia", "generator_id", "pollution_control_costs_per_kw"]]


merged = (
self.pudl_tabl.gens_eia860()
.query("operational_status == 'existing'")
.assign(
    prime_mover=lambda x: x.prime_mover_code.replace(
        FOSSIL_PRIME_MOVER_MAP
    ),
)
.copy()
.merge(
    coi[["plant_id_eia", "generator_id", "pollution_control_costs_per_kw"]],
    on=["plant_id_eia", "generator_id"],
    how="left",
    validate="m:1",
)
.fillna({"pollution_control_costs_per_kw": 0.0})
.merge(
    self.pudl_tabl.gens_eia860m()
    .query("report_date == report_date.max()")[
        [
            "plant_id_eia",
            "generator_id",
            "balancing_authority_code_eia",
            # "state",
        ]
    ]
    .drop_duplicates(),
    on=["plant_id_eia", "generator_id",],
    how="left",
    # indicator=True,
    validate="m:1",
)
.assign(
    age_in_report_year=lambda x: (
        x["report_date"] - x["generator_operating_date"]
    ).dt.days
    / 365.25,
    age_in_current_year=lambda x: (
        reference_date - x["generator_operating_date"]
    ).dt.days
    / 365.25,
    age_of_observation=lambda x: (reference_date - x["report_date"]).dt.days
    / 365.25,
    age_relative_to_prime_avg=lambda x: x["age_in_report_year"]
    - x.groupby(["prime_mover"])["age_in_report_year"].transform("mean"),
)  # filter out when generator operating date is na for pandera
.query("age_in_report_year.notna()")
)

/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(


In [125]:
merged.query('plant_id_eia == 136',engine='python')

,report_date,plant_id_eia,plant_id_pudl,plant_name_eia,utility_id_eia,utility_id_pudl,utility_name_eia,generator_id,associated_combined_heat_power,bga_source,bypass_heat_recovery,capacity_mw,carbon_capture,city,cofire_fuels,county,current_planned_generator_operating_date,data_maturity,deliver_power_transgrid,distributed_generation,duct_burners,energy_source_1_transport_1,energy_source_1_transport_2,energy_source_1_transport_3,energy_source_2_transport_1,energy_source_2_transport_2,energy_source_2_transport_3,energy_source_code_1,energy_source_code_2,energy_source_code_3,energy_source_code_4,energy_source_code_5,energy_source_code_6,energy_storage_capacity_mwh,ferc_qualifying_facility,fluidized_bed_tech,fuel_type_code_pudl,fuel_type_count,generator_operating_date,generator_retirement_date,latitude,longitude,minimum_load_mw,multiple_fuels,nameplate_power_factor,net_capacity_mwdc,operating_switch,operational_status,operational_status_code,original_planned_generator_operating_date,other_combustion_tech,other_modifications_date,other_planned_modifications,owned_by_non_utility,ownership_code,planned_derate_date,planned_energy_source_code_1,planned_generator_retirement_date,planned_modifications,planned_net_summer_capacity_derate_mw,planned_net_summer_capacity_uprate_mw,planned_net_winter_capacity_derate_mw,planned_net_winter_capacity_uprate_mw,planned_new_capacity_mw,planned_new_prime_mover_code,planned_repower_date,planned_uprate_date,previously_canceled,prime_mover_code,pulverized_coal_tech,reactive_power_output_mvar,rto_iso_lmp_node_id,rto_iso_location_wholesale_reporting_id,solid_fuel_gasification,startup_source_code_1,startup_source_code_2,startup_source_code_3,startup_source_code_4,state,stoker_tech,street_address,subcritical_tech,summer_capacity_estimate,summer_capacity_mw,summer_estimated_capability_mw,supercritical_tech,switch_oil_gas,syncronized_transmission_grid,technology_description,time_cold_shutdown_full_load_code,timezone,topping_bottoming_code,turbines_inverters_hydrokinetics,turbines_num,ultrasupercritical_tech,unit_id_pudl,uprate_derate_completed_date,uprate_derate_during_year,winter_capacity_estimate,winter_capacity_mw,winter_estimated_capability_mw,zip_code,prime_mover,pollution_control_costs_per_kw,balancing_authority_code_eia,age_in_report_year,age_in_current_year,age_of_observation,age_relative_to_prime_avg
290,2001-01-01,136,1310,Seminole,21554,3118,"Seminole Electric Coop, Inc",1,False,<NA>,False,714.6,<NA>,Palatka,<NA>,Putnam,NaT,final,<NA>,False,False,WT,<NA>,<NA>,<NA>,<NA>,<NA>,BIT,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,False,<NA>,coal,1,1984-02-01,NaT,29.733056,-81.63278,NaN,<NA>,NaN,NaN,<NA>,existing,OP,NaT,<NA>,NaT,<NA>,<NA>,S,NaT,<NA>,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaT,NaT,<NA>,ST,True,NaN,<NA>,<NA>,False,<NA>,<NA>,<NA>,<NA>,FL,<NA>,State Highway 19,True,<NA>,658.0,NaN,<NA>,<NA>,<NA>,Conventional Steam Coal,<NA>,America/New_York,X,<NA>,<NA>,<NA>,<NA>,NaT,<NA>,<NA>,665.0,NaN,32708,ST,1.289394,SEC,16.917180,37.831622,20.914442,-21.260704
16148,2002-01-01,136,1310,Seminole,21554,3118,"Seminole Electric Coop, Inc",1,False,<NA>,False,714.6,<NA>,Palatka,<NA>,Putnam,NaT,final,<NA>,<NA>,False,RR,<NA>,<NA>,<NA>,<NA>,<NA>,BIT,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,False,<NA>,coal,1,1984-02-01,NaT,29.733056,-81.63278,NaN,<NA>,NaN,NaN,<NA>,existing,OP,NaT,<NA>,NaT,<NA>,<NA>,S,NaT,<NA>,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaT,NaT,<NA>,ST,True,NaN,<NA>,<NA>,False,<NA>,<NA>,<NA>,<NA>,FL,<NA>,State Highway 19,True,<NA>,658.0,NaN,<NA>,<NA>,<NA>,Conventional Steam Coal,<NA>,America/New_York,X,<NA>,0,<NA>,<NA>,NaT,<NA>,<NA>,665.0,NaN,32708,ST,1.289394,SEC,17.916496,37.831622,19.915127,-20.261389
32566,2003-01-01,136,1310,Seminole,21554,3118,"Seminole Electric Coop, Inc",1,False,<NA>,False,714.6,<NA>,Palatka,<NA>,Putnam,NaT,final,<NA>,False,False,RR,<NA>,<NA>,<NA>,<NA>,<NA>,BIT,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,False,<NA>,coal,1,1984-02-01,NaT,29.733056,-81.63278,NaN,<NA>,NaN,NaN,<NA>,existing,OP,NaT,<NA>,NaT,<NA>,<NA>,S,NaT,<NA>,NaT,<NA>,NaN,N

In [123]:
self.get_860_by_x(subplant_id_col='generator_id')

/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(


,utility_id_eia,balancing_authority_code_eia,state,plant_id_eia,generator_id,report_date,capacity_mw,prime_mover,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_in_current_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,generator_operating_date,technology_description,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code
0,195,SOCO,AL,2,1,2001-01-01,45.0,HY,1,1,1,1,1,1,1,1,1,1,1,1,37.505818,58.420260,20.914442,-22.507147,2.92732,1963-07-01,Conventional Hydroelectric,2,<NA>,2,2
1,195,SOCO,AL,3,1,2001-01-01,153.1,ST,1,1,1,1,1,1,1,1,1,1,1,1,46.915811,67.830253,20.914442,8.737927,2.92732,1954-02-01,Conventional Steam Coal,2,<NA>,2,2
2,195,SOCO,AL,3,2,2001-01-01,153.1,ST,1,1,1,1,1,1,1,1,1,1,1,1,46.505133,67.419576,20.914442,8.327249,2.92732,1954-07-01,Conventional Steam Coal,2,<NA>,2,2
3,195,SOCO,AL,3,3,2001-01-01,272.0,ST,1,1,1,1,1,1,1,1,1,1,1,1,41.505818,62.420260,20.914442,3.327933,0.00000,1959-07-01,Conventional Steam Coal,2,<NA>,2,2
4,195,SOCO,AL,3,4,2001-01-01,403.7,ST,1,1,1,1,1,1,1,1,1,1,1,1,31.085558,52.000000,20.914442,-7.092327,2.92732,1969-12-01,Conventional Steam Coal,2,<NA>,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422452,65056,NYIS,NY,65815,MPRES,2022-01-01,1.0,PV,1,1,1,1,1,1,1,1,1,1,1,1,0.084873,0.000000,-0.084873,-2.982862,0.00000,2021-12-01,Solar Photovoltaic,<NA>,<NA>,<NA>,NYIS
422453,65069,DUK,NC,65817,CLEA1,2022-01-01,3.6,PV,1,1,1,1,1,1,1,1,1,1,1,1,1.333333,1.248460,-0.084873,-1.734402,0.00000,2020-09-01,Solar Photovoltaic,<NA>,<NA>,<NA>,DUKE
422454,61980,NaN,CA,65824,VSPRC,2022-01-01,1.1,PV,1,1,1,1,1,1,1,1,1,1,1,1,1.752225,1.667351,-0.084873,-1.315511,0.00000,2020-04-01,Solar Photovoltaic,<NA>,<NA>,<NA>,NaN
422455,65076,ERCO,TX,65836,TOYAH,2022-01-01,10.4,BA,1,1,1,1,1,1,1,1,1,1,1,1,0.251882,0.167009,-0.084873,-1.996059,0.00000,2021-10-01,Batteries,<NA>,<NA>,<NA>,ERCO
